# Healthy-control pseudostructure reconstruction versus DPT and scFates

This biological experiment uses **actual Ctrl1A2 and Ctrl1A4 sections**, not simulated positions. Infer from marker-excluded genes in one specimen, freeze preprocessing and coordinates, and predict held-out genes in the other. Swap specimens and rotate three fixed gene folds. Thousands of sections represent partial observations of repeated PT structures; unknown nephron membership remains integrated into the residual model. The mice are the replication units.

The primary component comparison gives DPT, scFates and each atlas start the **same frozen neighbor projector**. Separately evaluate native Gaussian atlas projection. These matched-input DPT/scFates refits differ from historical pooled-Harmony notebook 13, which included marker and query information. No true longitudinal position is available; depth is excluded from fitting, scoring and model selection. See the methods survey and research ledger for the hypotheses specified before outputs.

In [ ]:
from pathlib import Path
import hashlib, io, json, sys
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pseudospace').is_dir())
sys.path.insert(0,str(REPO))
from pseudospace.repeated_inputs import load_repeated_mouse_inputs
from pseudospace.count_representation import fit_count_representation,transform_count_representation
from pseudospace.joint_repeated_atlas import fit_joint_atlas,project_joint_atlas
from pseudospace.repeated_structure import fit_atlas
from pseudospace.repeated_atlas_baselines import fit_pc1_reference,project_pc1_reference,fit_segment_mixture,predict_segment_mixture
from pseudospace.within_segment_validation import (fit_segment_covariate_oracle,predict_segment_covariate_oracle,permute_positions_within_segments)
from pseudospace.stage_cache import cached_payload
from pseudospace.trajectory_benchmark import fit_dpt_axis,fit_scfates_axis,project_neighbor_axis
from pseudospace.atlas_initialization import initialize_atlas_positions
from importlib.metadata import version
import argparse,os
parser=argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root',default=os.environ.get('PSEUDOSPACE_DATA_ROOT',str(REPO/'data')))
parser.add_argument('--results-root',default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT',str(REPO/'results')))
args,_=parser.parse_known_args()
DATA_ROOT,RESULTS_ROOT=Path(args.data_root).expanduser(),Path(args.results_root).expanduser()
OUT=RESULTS_ROOT/'healthy_reconstruction_trajectory_benchmark'
OUT.mkdir(parents=True,exist_ok=True)
CONTROLS=('Ctrl1A2','Ctrl1A4')
NOTEBOOK_LOGIC_VERSION='21.healthy_trajectory_benchmark.1'
SEED,N_COMPONENTS,N_FOLDS=15,15,3
INITIALIZERS=('pc1','endpoint_contrast','soft_segment')
VERSIONS={name:version(name) for name in ['scanpy','scFates','anndata','scikit-learn']}
GRID=np.linspace(0,1,101)
MODEL_PARAMS=dict(grid=GRID,roughness=.001,offset_penalty=4.,anatomy_strength=1.,anatomy_width=.1,max_iter=60,tol=1e-5)
MARKERS=set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Slc7a12 Lrp2 Cubn Slc34a1 Slc6a18 Acsm3 Agt'.casefold().split())
CODE_FILES=[REPO/'pseudospace'/f for f in ['repeated_inputs.py','count_representation.py','joint_repeated_atlas.py','repeated_structure.py','repeated_atlas_baselines.py','within_segment_validation.py','trajectory_benchmark.py','atlas_initialization.py']]
CODE_DIGEST=hashlib.sha256(b''.join(p.read_bytes() for p in CODE_FILES)).hexdigest()
CACHE_PARAMS=dict(version=NOTEBOOK_LOGIC_VERSION,controls=CONTROLS,seed=SEED,
    components=N_COMPONENTS,folds=N_FOLDS,markers=sorted(MARKERS),
    model={**MODEL_PARAMS,'grid':GRID.tolist()},initializers=INITIALIZERS,
    dpt_neighbors=30,scfates_nodes=30,projection_neighbors=15,
    gene_spline_ridge=.001,oracle_ridge=.01,classifier_C=1.,versions=VERSIONS)
print('Installed benchmark versions:',VERSIONS)

## Validated healthy inputs and frozen gene folds

Validate kept-fine GeoJSON joins and centroids, apply the selected-cohort pooled 5th-percentile structural spot floor, and retain common measured genes. Do no tubule gene-count filter. Raw all-gene exposure includes held-out and marker genes, so compositional dependence remains. Training coarse labels retain upstream marker provenance; marker recovery is a supporting diagnostic, not wholly independent anatomy.

In [ ]:
healthy=load_repeated_mouse_inputs(DATA_ROOT,samples=CONTROLS)
obs=healthy.obs.copy()
ids=obs.structure_id.to_numpy(str)
specimens=obs['sample'].astype(str).to_numpy()
labels=obs.segment_class.astype(str).str.replace('PT-','',regex=False).to_numpy()
if set(np.unique(specimens))!=set(CONTROLS): raise ValueError('Expected exactly the two healthy control specimens')
if not np.isin(labels,['S1','S2','S3']).all(): raise ValueError('Resolve uncertain PT labels upstream')
anatomy=np.array([{'S1':0,'S2':1,'S3':2}[x] for x in labels],dtype=int)
genes=healthy.var_names.to_numpy(str)
X=healthy.layers['lognorm']; X=X.toarray().astype(np.float64,copy=False) if sparse.issparse(X) else np.asarray(X,dtype=np.float64)
raw_counts=healthy.layers['counts']
raw_total=np.asarray(raw_counts.sum(axis=1)).ravel()
if not np.isfinite(X).all() or not np.isfinite(raw_total).all() or np.any(raw_total<=0): raise ValueError('Invalid expression or all-gene exposure')
if 'n_spots' not in obs: raise ValueError('Required n_spots metadata is missing')
n_spots=pd.to_numeric(obs.n_spots,errors='coerce').to_numpy(float)
if not np.isfinite(n_spots).all() or np.any(n_spots<0): raise ValueError('Invalid n_spots metadata')
marker_mask=np.array([g.casefold() in MARKERS for g in genes])
gene_fold=np.array([int(hashlib.sha256(g.casefold().encode()).hexdigest()[:8],16)%N_FOLDS for g in genes])
manifest=dict(healthy.uns['repeated_mouse_input_manifest'])
manifest.update(notebook_logic_version=NOTEBOOK_LOGIC_VERSION,config=CACHE_PARAMS,code_digest=CODE_DIGEST,markers_excluded=sorted(MARKERS),dpt_used_for_fitting=False,depth_used=False)
(OUT/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
display(pd.crosstab(specimens,labels))
print('Structures:',len(ids),'genes:',len(genes),'outer split: whole specimen')

## Common prediction and evaluation

Training-only count residual PCs (15 components, dispersion 100) are identical for all methods in a transfer/fold. DPT is rooted in training S1; scFates uses a nonbranching 30-node curve and an S1-oriented tip. Both receive endpoint orientation but no fitting likelihood using all labels. The atlas additionally uses weak ordinal labels in its likelihood; differences in supervision are disclosed.

Every continuous coordinate predicts response genes with the same spline/ridge and training-variance normalization. The shared query extension uses 15 inverse-distance-weighted training neighbors; it is a benchmark extension, not a native DPT/scFates API. The native Gaussian comparison uses the same atlas training axis and an expression-only posterior. The soft segment predictor accepts query expression only; the deliberately privileged coverage oracle receives query labels for diagnosis.

In [ ]:
def dense_panel(matrix,rows,columns):
    value=matrix[rows][:,columns]
    return value.toarray() if sparse.issparse(value) else np.asarray(value)

def orient_pca(Y,a):
    value=Y[:,0].copy()
    if np.median(value[a==2])<np.median(value[a==0]): value=-value
    lo,hi=np.quantile(value,[.01,.99])
    if hi<=lo: raise ValueError('Degenerate training PC1 initializer')
    return np.clip((value-lo)/(hi-lo),0,1)

def project_gaussian(atlas,Y,offset):
    return project_joint_atlas(atlas,Y,offset=offset)

def fit_gene_spline(response,z,sample):
    return fit_atlas(response,z,sample,grid=GRID,ridge=.001)

def normalized_variance(response,sample):
    mean=np.mean([response[sample==s].mean(axis=0) for s in np.unique(sample)],axis=0)
    return np.maximum(np.mean([((response[sample==s]-mean)**2).mean(axis=0) for s in np.unique(sample)],axis=0),1e-3)

def primary_rows(response,predictions,variance,query_anatomy,meta):
    out=[]
    for method,pred in predictions.items():
        scaled=(response-pred)**2/variance
        for seg,name in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
            use=np.ones(len(response),bool) if seg is None else query_anatomy==seg
            if use.any(): out.append(dict(**meta,method=method,segment=name,n=int(use.sum()),normalized_mse=float(scaled[use].mean())))
    return out

In [ ]:
def run_healthy_benchmark():
    score_rows,marker_rows,position_rows,diagnostics,failures=[],[],[],[],[]
    for source in CONTROLS:
        target=next(s for s in CONTROLS if s!=source)
        tr,te=specimens==source,specimens==target
        for fold in range(N_FOLDS):
            key=dict(source_specimen=source,test_specimen=target,fold=fold)
            mg=np.flatnonzero(~marker_mask & (gene_fold!=fold))
            eg=np.flatnonzero(~marker_mask & (gene_fold==fold))
            train_counts=dense_panel(raw_counts,tr,mg);query_counts=dense_panel(raw_counts,te,mg)
            Y,transform=fit_count_representation(train_counts,raw_total[tr],specimens[tr],
                N_COMPONENTS,theta=100.,seed=SEED)
            query_Y=transform_count_representation(query_counts,raw_total[te],transform)
            train_response=X[tr][:,eg];query_response=X[te][:,eg]
            variance=normalized_variance(train_response,specimens[tr])
            coordinates={};predictions={}
            # Baselines and candidates share representation; each failure is explicit.
            for name,fit_function,params in [('DPT',fit_dpt_axis,dict(n_neighbors=30,seed=SEED)),
                    ('scFates',fit_scfates_axis,dict(nodes=30,seed=SEED))]:
                try:
                    train_z,meta=fit_function(Y,anatomy[tr],**params)
                    query_z=project_neighbor_axis(Y,train_z,query_Y,k=15)
                    method=name+'_neighbor'
                    coordinates[method]=query_z
                    predictions[method]=fit_gene_spline(train_response,train_z,specimens[tr]).predict(query_z)
                    diagnostics.append(dict(**key,method=name,status='complete',metadata=json.dumps(meta)))
                    marker_indices=np.flatnonzero(marker_mask)
                    marker_prediction=fit_gene_spline(X[tr][:,marker_indices],train_z,specimens[tr]).predict(query_z)
                    marker_rows.extend(primary_rows(X[te][:,marker_indices],{method:marker_prediction},
                        normalized_variance(X[tr][:,marker_indices],specimens[tr]),anatomy[te],key))
                except (ValueError,RuntimeError,np.linalg.LinAlgError) as exc:
                    failures.append(dict(**key,method=name,error=str(exc)))
                    diagnostics.append(dict(**key,method=name,status='failed',metadata=str(exc)))
            for initializer in INITIALIZERS:
                name='atlas_'+initializer
                try:
                    initial,initial_meta=initialize_atlas_positions(Y,anatomy[tr],specimens[tr],method=initializer,seed=SEED)
                    atlas=fit_joint_atlas(Y,specimens[tr],anatomy[tr],initial_z=initial,**MODEL_PARAMS)
                    info=dict(converged=bool(atlas['converged']),iterations=len(atlas['history']),
                        objective=atlas['history'][-1]['objective'],initializer=initializer)
                    if not atlas['converged']:
                        diagnostics.append(dict(**key,method=name,status='failed',metadata=json.dumps(info)))
                        raise RuntimeError('joint Gaussian atlas did not converge')
                    train_z=project_joint_atlas(atlas,Y,offset=atlas['offsets'][source])['z_mean']
                    decoder=fit_gene_spline(train_response,train_z,specimens[tr])
                    query_coordinates=dict(neighbor=project_neighbor_axis(Y,train_z,query_Y,k=15),
                        native=project_joint_atlas(atlas,query_Y,offset=np.zeros(Y.shape[1]))['z_mean'])
                    marker_indices=np.flatnonzero(marker_mask)
                    marker_decoder=fit_gene_spline(X[tr][:,marker_indices],train_z,specimens[tr])
                    for projector,query_z in query_coordinates.items():
                        method=name+'_'+projector
                        coordinates[method]=query_z
                        predictions[method]=decoder.predict(query_z)
                        marker_rows.extend(primary_rows(X[te][:,marker_indices],{method:marker_decoder.predict(query_z)},
                            normalized_variance(X[tr][:,marker_indices],specimens[tr]),anatomy[te],key))
                    diagnostics.append(dict(**key,method=name,status='complete',metadata=json.dumps(info)))
                except (ValueError,RuntimeError,np.linalg.LinAlgError) as exc:
                    failures.append(dict(**key,method=name,error=str(exc)))
                    if not any(row['method']==name and all(row[k]==v for k,v in key.items()) for row in diagnostics):
                        diagnostics.append(dict(**key,method=name,status='failed',metadata=str(exc)))
            pc=fit_pc1_reference(Y,anatomy[tr])
            pc_train=project_pc1_reference(pc,Y);pc_query=project_pc1_reference(pc,query_Y)
            coordinates['count_PC1']=pc_query
            predictions['count_PC1']=fit_gene_spline(train_response,pc_train,specimens[tr]).predict(pc_query)
            mixture=fit_segment_mixture(Y,train_response,anatomy[tr],specimens[tr],C=1.,seed=SEED)
            predictions['soft_segment']=predict_segment_mixture(mixture,query_Y)['prediction']
            detected_train=np.asarray((raw_counts[tr][:,mg]>0).sum(axis=1)).ravel()
            detected_query=np.asarray((raw_counts[te][:,mg]>0).sum(axis=1)).ravel()
            train_cov=np.column_stack([np.log(raw_total[tr]),np.log1p(n_spots[tr]),np.log1p(detected_train)])
            query_cov=np.column_stack([np.log(raw_total[te]),np.log1p(n_spots[te]),np.log1p(detected_query)])
            oracle=fit_segment_covariate_oracle(train_response,train_cov,anatomy[tr],specimens[tr],ridge=.01)
            predictions['segment_coverage_oracle']=predict_segment_covariate_oracle(oracle,query_cov,anatomy[te])
            score_rows.extend(primary_rows(query_response,predictions,variance,anatomy[te],key))
            for method,z in coordinates.items():
                position_rows.extend(dict(**key,method=method,structure_id=id_,z=float(value),
                    segment=label) for id_,value,label in zip(ids[te],z,labels[te]))
            print('Completed actual-data transfer:',source,'→',target,'fold',fold,flush=True)
    return {name:pd.DataFrame(rows).to_json(orient='table') for name,rows in dict(scores=score_rows,
        marker_scores=marker_rows,positions=position_rows,diagnostics=diagnostics,failures=failures).items()}

In [ ]:
CACHE_INPUTS=dict(X=X,counts=raw_counts,library=raw_total,ids=ids,specimens=specimens,
    labels=labels,n_spots=n_spots,gene_fold=gene_fold,markers=marker_mask)
payload=cached_payload('healthy_reconstruction_trajectory_benchmark',run_healthy_benchmark,
    root=OUT/'stage_cache',params=CACHE_PARAMS,inputs=CACHE_INPUTS,code=CODE_DIGEST)
outputs={name:pd.read_json(io.StringIO(str(value)),orient='table') for name,value in payload.items()}
for name,frame in outputs.items():frame.to_csv(OUT/(name+'.csv'),index=False)
# Guards and failure displays run on cache hits too.
if len(outputs['diagnostics'])!=len(CONTROLS)*N_FOLDS*(2+len(INITIALIZERS)):
    raise RuntimeError('Incomplete expected 30 algorithm-attempt records')
display(outputs['failures'])
display(outputs['diagnostics'][['source_specimen','fold','method','status']])
print('Failed attempts:',len(outputs['failures']),'of',len(outputs['diagnostics']))

## Primary axis-fitting comparison and secondary projector comparison

The primary table compares `_neighbor` coordinates through an identical frozen extension. Native atlas projection is secondary and uses no query labels. Within-segment error is molecular prediction, not true longitudinal location. All genes in a response fold remain included; no gene list or parameter is selected from these query scores. The existing two mice have already informed method development, so this is repeated descriptive benchmarking rather than a fresh external validation cohort.

In [ ]:
scores=outputs['scores']
display(scores.pivot_table(index=['source_specimen','fold','segment'],columns='method',values='normalized_mse'))
display(outputs['marker_scores'].query("segment=='all'").pivot_table(index=['source_specimen','fold'],columns='method',values='normalized_mse'))
comparison=scores.pivot_table(index=['source_specimen','test_specimen','fold','segment'],columns='method',values='normalized_mse')
relative=comparison.apply(lambda column:100*(comparison['scFates_neighbor']-column)/comparison['scFates_neighbor'])
display(relative.groupby(['source_specimen','segment']).mean())
relative.to_csv(OUT/'relative_gain_vs_scfates.csv')
import matplotlib.pyplot as plt
primary=['DPT_neighbor','scFates_neighbor','atlas_pc1_neighbor','atlas_endpoint_contrast_neighbor','atlas_soft_segment_neighbor']
fig,axes=plt.subplots(1,2,figsize=(12,4),sharey=True)
for ax,source in zip(axes,CONTROLS):
    panel=scores.query("segment=='all' and source_specimen==@source")
    for j,method in enumerate(primary):
        values=relative.xs(source,level='source_specimen').xs('all',level='segment')[method].dropna().to_numpy()
        ax.scatter(np.full(len(values),j),values,color='C'+str(j),s=25)
        if len(values):ax.plot([j-.18,j+.18],[values.mean()]*2,color='C'+str(j))
    ax.set_xticks(range(len(primary)),[x.replace('_neighbor','') for x in primary],rotation=30,ha='right',fontsize=8)
    ax.set_title(source+' → held-out control')
    ax.axhline(0,color='grey',linewidth=.7)
    failed=outputs['failures'].query('source_specimen==@source')
    ax.text(.01,.02,str(len(failed))+' failed atlas starts',transform=ax.transAxes,fontsize=8)
axes[0].set_ylabel('Held-out gene error reduction vs scFates (%)')
fig.suptitle('Shared query projector; dots are gene folds, not biological replicates',fontsize=10)
fig.tight_layout();plt.show()

## Measured conclusion

All six DPT and six scFates fits complete; PC1 and endpoint atlas starts also converge in all six transfers. Soft starts fail in Ctrl1A2 folds 0 and 2, so their means do not represent complete three-fold comparisons. The shared-projector scFates backbone improves whole-gene error over DPT by approximately 0.28% in Ctrl1A2→Ctrl1A4 and is almost tied in the reverse direction. The joint atlas does not consistently outperform it: native PC1 projection is approximately 0.047% worse in the first direction and 0.022% better in the reverse. Small native-vs-neighbor gains do not establish anatomical accuracy.

Endpoint and PC1 fits are nearly equivalent in actual data. The stronger marker prediction under the ordinal atlas cannot count as wholly independent validation because training labels retain marker provenance. The segment-plus-coverage oracle explains more of the global molecular prediction in both directions, and segment-specific comparisons vary. These are useful constraints on the model, not a reason to select an initializer from query scores.

The next biological hypothesis is a simpler repeated-structure reference built on an established backbone: estimate conditional healthy molecular distributions and test frozen count-aware projection, rather than requiring joint latent curve fitting to outperform scFates. Literature supports using established backbone and registration tools. Transport remains optional and must show incremental value.